In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(0)

Data preparation

c = 3e8
fc = 3.5e9
d0 = 1.0
FSPL_d0_theory = 20 * np.log10(4 * np.pi * d0 * fc / c)

def synthesize_pathloss(n_true=3.0, sigma_true=6.0, n_samples=150, seed=1):
    rng = np.random.default_rng(seed)
    log_d = rng.uniform(np.log10(10), np.log10(1800), n_samples)
    d = 10 ** log_d
    shadowing = rng.normal(0, sigma_true, n_samples)
    pl = FSPL_d0_theory + 10 * n_true * np.log10(d / d0) + shadowing
    return d, pl

path = Path("pathloss.txt")
if path.exists():
    data = np.loadtxt(path)
    d_meas = data[:, 0]
    PL_meas = data[:, 1]
else:
    d_meas, PL_meas = synthesize_pathloss()

Regression model

In [ ]:
class MyLinearRegression:
    def __init__(self):
        self.theta = None

    @staticmethod
    def _design_matrix(x):
        return np.column_stack([np.ones(len(x)), x])

    def fit(self, x, y):
        Phi = self._design_matrix(x)
        self.theta = np.linalg.solve(Phi.T @ Phi, Phi.T @ y)
        return self

    def predict(self, x):
        return self._design_matrix(x) @ self.theta

Train model

In [ ]:
x_feat = np.log10(d_meas / d0)
y_target = PL_meas

model = MyLinearRegression().fit(x_feat, y_target)

Visualization

In [ ]:
d_grid = np.logspace(np.log10(d_meas.min()), np.log10(d_meas.max()), 200)
x_grid = np.log10(d_grid / d0)

plt.figure(figsize=(8, 5))
plt.scatter(x_feat, y_target, s=12, alpha=0.4, label="Measured path loss")
plt.plot(x_grid, model.predict(x_grid), linewidth=2,
         label="Linear Regression Fit")

plt.xlabel("log10(d/d0)")
plt.ylabel("Path Loss (dB)")
plt.title("Question 4: Path Loss Model Visualization")
plt.legend()
plt.tight_layout()
plt.show()